In [1]:
import os
import math
import random
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torchvision.transforms.functional as TF
from tqdm import tqdm


# =========================
# CONFIG
# =========================
@dataclass
class CFG:
    src_root: str = "dataset/original/train"
    tgt_root: str = "dataset/original/val"   # used UNLABELED during training
    img_size: int = 224

    batch_size: int = 16
    num_workers: int = 4
    epochs: int = 20
    lr: float = 3e-5
    weight_decay: float = 1e-4

    # loss weights
    lambda_domain: float = 0.5
    lambda_rot: float = 0.2

    # schedule for domain strength
    use_schedule: bool = True

    # model
    backbone: str = "efficientnet_b0"  # resnet18 / resnet34 / efficientnet_b0
    num_classes: int = 2

cfg = CFG()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# TRANSFORMS
# Keep simple & consistent; DANN handles shift via adversarial alignment.
# =========================
def get_tf_train(img_size=224):
    return transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((img_size, img_size)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
    ])

def get_tf_eval(img_size=224):
    return transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])


# =========================
# ROTATION TASK (self-supervised)
# =========================
def random_rotate_batch(x):
    """
    x: Tensor [B,C,H,W]
    returns: rotated_x, rot_labels (0,1,2,3) -> 0°,90°,180°,270°
    """
    B = x.size(0)
    rot_labels = torch.randint(0, 4, (B,), device=x.device)
    x_rot = x.clone()
    for i in range(B):
        k = int(rot_labels[i].item())
        x_rot[i] = torch.rot90(x_rot[i], k=k, dims=(1,2))
    return x_rot, rot_labels


# =========================
# GRADIENT REVERSAL LAYER
# =========================
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha):
    return GradReverse.apply(x, alpha)


# =========================
# MODEL: Backbone + Heads
# =========================
class DANNMultiTask(nn.Module):
    def __init__(self, backbone="efficientnet_b0", num_classes=2):
        super().__init__()

        if backbone == "resnet18":
            m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            feat_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m

        elif backbone == "resnet34":
            m = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
            feat_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m

        elif backbone == "efficientnet_b0":
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            feat_dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            self.backbone = m

        else:
            raise ValueError("Unsupported backbone")

        # Task A: crack classification
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(feat_dim, num_classes),
        )

        # Task B: domain classifier (source vs target)
        self.domain = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )

        # Task C: rotation classifier (self-supervised)
        self.rot = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 4),
        )

    def forward_features(self, x):
        f = self.backbone(x)
        # EfficientNet returns [B,feat] already; ResNet returns [B,feat]
        return f

    def forward(self, x, alpha=1.0):
        feat = self.forward_features(x)

        # classification head
        y_cls = self.classifier(feat)

        # domain head with gradient reversal
        feat_rev = grad_reverse(feat, alpha)
        y_dom = self.domain(feat_rev)

        # rotation head (no reversal)
        y_rot = self.rot(feat)

        return y_cls, y_dom, y_rot


# =========================
# ALPHA SCHEDULE (standard DANN schedule)
# =========================
def dann_alpha(progress):
    # progress in [0,1]
    return float(2.0 / (1.0 + math.exp(-10 * progress)) - 1.0)


# =========================
# TRAIN / EVAL
# =========================
@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        y_cls, _, _ = model(x, alpha=0.0)
        pred = y_cls.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / max(total, 1)


def train_dann(model, src_loader, tgt_loader, val_eval_loader):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    ce = nn.CrossEntropyLoss()

    best = 0.0
    os.makedirs("runs_dann", exist_ok=True)

    # iterate both loaders together
    steps_per_epoch = min(len(src_loader), len(tgt_loader))
    global_step = 0
    total_steps = cfg.epochs * steps_per_epoch

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch}", unit="step")
        for _ in pbar:
            xs, ys = next(src_iter)
            xt, _  = next(tgt_iter)

            xs, ys = xs.to(DEVICE), ys.to(DEVICE)
            xt = xt.to(DEVICE)

            # DANN alpha schedule
            if cfg.use_schedule:
                progress = global_step / max(total_steps, 1)
                alpha = dann_alpha(progress)
            else:
                alpha = 1.0

            # forward: source + target for domain
            ycls_s, ydom_s, yrot_s = model(xs, alpha=alpha)
            ycls_t, ydom_t, yrot_t = model(xt, alpha=alpha)

            # ---- Loss A: classification on SOURCE only
            loss_cls = ce(ycls_s, ys)

            # ---- Loss B: domain classification on SOURCE+TARGET
            dom_s = torch.zeros(xs.size(0), dtype=torch.long, device=DEVICE)
            dom_t = torch.ones(xt.size(0), dtype=torch.long, device=DEVICE)
            loss_dom = ce(ydom_s, dom_s) + ce(ydom_t, dom_t)

            # ---- Loss C: rotation self-supervision on both domains (optional but strong)
            xs_rot, rs = random_rotate_batch(xs)
            xt_rot, rt = random_rotate_batch(xt)
            _, _, yrot_s2 = model(xs_rot, alpha=alpha)
            _, _, yrot_t2 = model(xt_rot, alpha=alpha)
            loss_rot = ce(yrot_s2, rs) + ce(yrot_t2, rt)

            loss = loss_cls + cfg.lambda_domain * loss_dom + cfg.lambda_rot * loss_rot

            opt.zero_grad()
            loss.backward()
            opt.step()

            # quick metrics (classification on source batch)
            with torch.no_grad():
                acc_s = (ycls_s.argmax(1) == ys).float().mean().item() * 100.0

            pbar.set_postfix({
                "alpha": f"{alpha:.2f}",
                "L": f"{loss.item():.3f}",
                "Lcls": f"{loss_cls.item():.3f}",
                "Ldom": f"{loss_dom.item():.3f}",
                "Lrot": f"{loss_rot.item():.3f}",
                "acc_s": f"{acc_s:.1f}",
            })

            global_step += 1

        # evaluate on target (using labels ONLY for evaluation)
        val_acc = evaluate_cls(model, val_eval_loader)
        print(f"[Epoch {epoch}] Target-val Acc: {val_acc:.2f}%")

        if val_acc > best:
            best = val_acc
            torch.save(model.state_dict(), "runs_dann/best_model.pth")
            print(f"Saved best: {best:.2f}%")

    print(f"Best target-val accuracy: {best:.2f}%")


def main():
    tf_train = get_tf_train(cfg.img_size)
    tf_eval  = get_tf_eval(cfg.img_size)

    # Source labeled
    src_ds = datasets.ImageFolder(cfg.src_root, transform=tf_train)
    src_loader = DataLoader(src_ds, batch_size=cfg.batch_size, shuffle=True,
                            num_workers=cfg.num_workers, pin_memory=True, drop_last=True)

    # Target unlabeled for training (ignore labels)
    tgt_ds = datasets.ImageFolder(cfg.tgt_root, transform=tf_train)
    tgt_loader = DataLoader(tgt_ds, batch_size=cfg.batch_size, shuffle=True,
                            num_workers=cfg.num_workers, pin_memory=True, drop_last=True)

    # Target labeled for evaluation ONLY
    tgt_eval = datasets.ImageFolder(cfg.tgt_root, transform=tf_eval)
    tgt_eval_loader = DataLoader(tgt_eval, batch_size=cfg.batch_size, shuffle=False,
                                 num_workers=cfg.num_workers, pin_memory=True)

    model = DANNMultiTask(backbone=cfg.backbone, num_classes=cfg.num_classes)
    train_dann(model, src_loader, tgt_loader, tgt_eval_loader)


if __name__ == "__main__":
    main()


Epoch 1: 100%|██████████| 422/422 [00:47<00:00,  8.92step/s, alpha=0.24, L=1.257, Lcls=0.201, Ldom=1.081, Lrot=2.581, acc_s=93.8] 


[Epoch 1] Target-val Acc: 66.62%
Saved best: 66.62%


Epoch 2: 100%|██████████| 422/422 [00:45<00:00,  9.33step/s, alpha=0.46, L=1.237, Lcls=0.118, Ldom=1.103, Lrot=2.838, acc_s=93.8] 


[Epoch 2] Target-val Acc: 76.47%
Saved best: 76.47%


Epoch 3: 100%|██████████| 422/422 [00:45<00:00,  9.30step/s, alpha=0.63, L=1.243, Lcls=0.102, Ldom=1.396, Lrot=2.216, acc_s=100.0]


...

[Epoch 18] Target-val Acc: 80.99%


Epoch 19: 100%|██████████| 422/422 [00:45<00:00,  9.33step/s, alpha=1.00, L=0.993, Lcls=0.001, Ldom=1.373, Lrot=1.529, acc_s=100.0]


[Epoch 19] Target-val Acc: 80.23%


Epoch 20: 100%|██████████| 422/422 [00:45<00:00,  9.28step/s, alpha=1.00, L=1.103, Lcls=0.043, Ldom=1.355, Lrot=1.914, acc_s=100.0]


[Epoch 20] Target-val Acc: 78.97%
Best target-val accuracy: 81.21%


### Task
Load the saved best DANN multitask model (`runs_dann/best_model.pth`) and evaluate it on the labeled validation set (`cfg.tgt_root`) using the same eval transforms, reporting accuracy (and a simple classification report/confusion matrix).

In [2]:
# Load best model checkpoint
import os
import torch

ckpt_path = "runs_dann/best_model.pth"
assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

model = DANNMultiTask(backbone=cfg.backbone, num_classes=cfg.num_classes).to(DEVICE)
state = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

print(f"Loaded checkpoint: {ckpt_path}")
print(f"Device: {DEVICE}")
print(f"Backbone: {cfg.backbone}, num_classes: {cfg.num_classes}")


Loaded checkpoint: runs_dann/best_model.pth
Device: cuda
Backbone: efficientnet_b0, num_classes: 2


In [3]:
# Build validation loader (labeled target validation) with eval transforms
from torch.utils.data import DataLoader
from torchvision import datasets

tf_eval = get_tf_eval(cfg.img_size)

val_ds = datasets.ImageFolder(cfg.tgt_root, transform=tf_eval)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

print(f"Val samples: {len(val_ds)} | Classes: {val_ds.classes}")


Val samples: 6758 | Classes: ['defect', 'no_defect']


In [4]:
# Evaluate on validation: accuracy + confusion matrix + classification report
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

all_preds, all_y = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        y_cls, _, _ = model(x, alpha=0.0)
        preds = y_cls.argmax(1)
        all_preds.append(preds.cpu().numpy())
        all_y.append(y.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_y = np.concatenate(all_y)

acc = (all_preds == all_y).mean() * 100.0
cm = confusion_matrix(all_y, all_preds)

print(f"Validation Accuracy: {acc:.2f}%")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(all_y, all_preds, target_names=val_ds.classes, digits=4))



Validation Accuracy: 81.21%
Confusion Matrix:
 [[3051  907]
 [ 363 2437]]

Classification Report:
               precision    recall  f1-score   support

      defect     0.8937    0.7708    0.8277      3958
   no_defect     0.7288    0.8704    0.7933      2800

    accuracy                         0.8121      6758
   macro avg     0.8112    0.8206    0.8105      6758
weighted avg     0.8253    0.8121    0.8135      6758

